# Training RT-DETR Cell Instance Detection

## Preparations
### Import required libraries

In [ ]:
import os
import sys
import time
import pickle
import json
from typing import Tuple, Union, List, Dict, Final

import numpy as np
from PIL import Image
import cv2
import torch
import pycocotools
from pycocotools.coco import COCO
from pycocotools import mask as coco_mask_util
import albumentations as A
sys.path.append('references/detection')

## Pre-processing configurations
The following configurations are used for pre-processing the images during the training. 

In [ ]:
# probability of adding random blur and salt-and-pepper or additive gaussian noise to the training images
# see the image Transforms section below
P_NOISE = 0.25

# the lower and upper bounds for random scaling the images (and annotated masks) for training augmentation
MIN_RANDOM_SCALE = 0.7
MAX_RANDOM_SCALE = 1.0

# model input image size (should be square)
MODEL_INPUT_SIZE: Final[int] = 640

SEGMENTATION_MODEL: Final[bool] = False

### Transforms
We are using `Albumentations` package for all image and annotation augmentations.  

In [ ]:
def get_transform(train: bool = True) -> A.core.composition.Compose:
    # no resizing is needed as the images are already prepared as 640 x 640 (the same as the model's input size)
    if train:
        # random noise addition and random scale as defined above, 
        # we call these before PILToTensor as these classes 
        # operates on PIL images
        trsfms = [
            A.RandomScale(scale_limit=(MIN_RANDOM_SCALE - 1.0, MAX_RANDOM_SCALE - 1.0), p=1.0),
            A.PadIfNeeded(min_height = MODEL_INPUT_SIZE, min_width = MODEL_INPUT_SIZE, position = 'random'),
            A.RandomRotate90(),
            A.Perspective(p=0.1),
            A.RandomBrightnessContrast(p=P_NOISE),
            A.HueSaturationValue(p=P_NOISE),
        ]        
    else:
        # test set (only convert the bboxes)
        trsfms = [A.NoOp()]

    return A.Compose(trsfms, bbox_params=A.BboxParams(format="coco", label_fields=["category"], clip=True, min_area=1))

## Data Model
The dataset class as well as the training and evaluation scripts are adopted from the RT-DETR fine-tuning tuturial Notebook here:  https://github.com/NielsRogge/Transformers-Tutorials/blob/master/RT-DETR/Fine_tune_RT_DETR_on_a_custom_dataset.ipynb. This Notebook provides the dataset class for the pre-processed annotated data (resized and cropped) as prepared for training our YOLO model (640 x 640 crops). 

NOTE: The labels (class IDs) for both YOLO and RT-DETR models starts with 0.

### Dataset for already pre-processed annotated data
This dataset is built by passing the location of the pre-processed images and bounding box folders (labels). The code below assumes the images and the corresponding annotation files use the same name. For each image with name `sample_name` (saved as .jpg or .png) under the images folder, there should be an annotation file with the same name and .txt extension (`sample_name.txt`) under the annotation folder. For a faster training, pre-process the images and use the dataset below.

In [ ]:
class CellMaskDataset(torch.utils.data.Dataset):
    def __init__(self, 
                 images_path: str,
                 annotations_path: str, 
                 instance_segmentation: bool,
                 percentage_to_expand_bbox_boundaries: float, 
                 transforms: A.core.composition.Compose) -> None:
        
        self.images_path = images_path
        self.annotations_path = annotations_path
        self.instance_segmentation = instance_segmentation
        self.transforms = transforms
        # load all images and masks
        # the assumption is the image and its mask annotation use the same name
        self.imgs = list(sorted(os.listdir(images_path)))
        self.annotations = list(sorted(os.listdir(annotations_path)))
        self.percentage_to_expand_bbox_boundaries = percentage_to_expand_bbox_boundaries
        

        if len(self.imgs) != len(self.annotations):
            print("[ERROR]: The list of images and annotations are not consistent")
            return
        
        for i, img_filename in enumerate(self.imgs):
            # drop the image/mask filename extension 
            # (anything after the last '.' in the filename is considered as extension)
            img_name = ".".join(img_filename.strip().split('.')[:-1])
            annots_name = ".".join(self.annotations[i].strip().split('.')[:-1])
            if img_name != annots_name:
                print("[ERROR]: Inconsistent annotations file :{} found for image file: {}".format(annots_name, img_name))
     

    def __getitem__(self, idx: int):

        # a unique image identifier
        image_id: int = idx
        # load images and masks
        img_path = os.path.join(self.images_path, self.imgs[idx])
        annots_path = os.path.join(self.annotations_path, self.annotations[idx])
        # read the image, do not change the format
        # the processed images are all 8-bit (bit-depth)
        # the model expect the image in RGB format, convert grayscale images to RGB
        img = np.array(Image.open(img_path).convert('RGB'))
            
        image_height, image_width = img.shape[:2]
        # annotations
        # boxes should be in COCO format (xtl, ytl, w, h) because this is the format the model and the
        # albumentations expect
        boxes: List[List[int]] = [] 
        labels: List[int] = []
        masks: List[np.ndarray] = []
        
        with open(annots_path,'r') as annot_file:
            num_annotations = 0
            for line in annot_file:
                fields = line.strip().split(' ')
                if self.instance_segmentation:
                    # convert the label to an integer from string
                    label = int(fields[0])
                    # convert the polygon points fron strings to floats
                    points = np.array([float(point) for point in fields[1:]])
                    # rearrange them in (x, y)
                    polygon_points = np.reshape(points, (int(len(points) / 2), 2)) * np.array([image_width, image_height], dtype=float)
                    # convert the points to integers and use CV2 contours format
                    polygon_points = np.expand_dims(polygon_points.astype(int), axis=1)
                    # bounding box of the mask contour
                    (xtl, ytl, w, h) = cv2.boundingRect(polygon_points)
                    # skip zero area boxes
                    if w <= 0 or h <= 0:
                        continue
                    xbr = xtl + w 
                    ybr = ytl + h
                    
                    # the mask
                    mask: np.ndarray = np.zeros((image_height, image_width), np.uint8)
                    # create the mask for the object (the mask value is set to 1 for the object)
                    cv2.drawContours(mask, [polygon_points], 0, 1, -1)
                    masks.append(mask)
                else:
                    (label, center_x, center_y, w, h) = fields
                    xtl = int((float(center_x) - float(w) / 2.0) * image_width)
                    ytl = int((float(center_y) - float(h) / 2.0) * image_height)
                    xbr = int((float(center_x) + float(w) / 2.0) * image_width)
                    ybr = int((float(center_y) + float(h) / 2.0) * image_height)
                    if xtl >= xbr or ytl >= ybr:
                        continue
                    # convert the label to an integer from string
                    label = int(label)

                # expand the bounding boxes if necessary
                delta_x = int(self.percentage_to_expand_bbox_boundaries * (xbr - xtl) / 2)
                delta_y = int(self.percentage_to_expand_bbox_boundaries * (ybr - ytl) / 2)
                # # expand by one pixel on each side at least to cover boundaries
                delta_x = max(1, delta_x)
                delta_y = max(1, delta_y)
            
                xtl = max(0, xtl - delta_x)
                ytl = max(0, ytl - delta_y)
                xbr = min(image_width, xbr + delta_x)
                ybr = min(image_height, ybr + delta_y)
                boxes.append([xtl, ytl, xbr - xtl, ybr - ytl])
                labels.append(label)
                num_annotations += 1
        
        # convert boxes to a numpy array
        boxes: np.ndarray = np.array(boxes)
        
        # apply augmentations, the second condition should not happen (all preprocessed images should at least have one object)
        if self.transforms and len(boxes) > 0:
            if self.instance_segmentation:
                # TODO: check to make sure the below mask transform would work
                transformed = self.transforms(image=img, bboxes=boxes, masks=masks, category=labels)
                img = transformed["image"]
                boxes = transformed["bboxes"]
                masks = transformed["masks"]
                labels = transformed["category"]
            else:
                transformed = self.transforms(image=img, bboxes=boxes, category=labels)
                img = transformed["image"]
                boxes = transformed["bboxes"]
                labels = transformed["category"]
        
        # reformat annotations
        annotations: List[dict] = []
        for i, bbox in enumerate(boxes):
            formatted_annotation = {
                "image_id": image_id,
                "category_id": labels[i],
                "bbox": bbox,
                "iscrowd": 0,
                "area": bbox[2] * bbox[3],
            }
            if self.instance_segmentation:
                formatted_annotation["mask"] = masks[i]
            annotations.append(formatted_annotation)

        return img, {"image_id": image_id, "annotations": annotations,}

    def __len__(self):
        return len(self.imgs)


#### A function to convert our dataset to COCO dataset format
This function is needed for efficient evaluation. Similar to the dataset class, the labels (class IDs) should start from 0. 

In [ ]:
from tqdm import tqdm
def convert_to_coco_api(images_path: str, 
                        annotations_path: str, 
                        instance_segmentation: bool = False,
                        percentage_to_expand_bbox_boundaries: float = 0.0):
    # load all images and annotations
    # the assumption is the image and its annotation use the same name
    imgs = list(sorted(os.listdir(images_path)))
    annotations = list(sorted(os.listdir(annotations_path)))
    
    if len(imgs) != len(annotations):
        print("[ERROR]: The list of images and masks are not consistent")
        return False, {}
    
    for i, img_filename in enumerate(imgs):
        # drop the image/mask filename extension 
        # (anything after the last '.' in the filename is considered as extension)
        img_name = ".".join(img_filename.strip().split('.')[:-1])
        annots_name = ".".join(annotations[i].strip().split('.')[:-1])
        if img_name != annots_name:
            print("[ERROR]: Inconsistent annotations file :{} found for image file: {}".format(annots_name, img_name))
            return False, {}
    # the index for annotations starts at 1
    annots_id = 1
    categories = set()
    json_annotations = {"images": [], "categories": [], "annotations": []}
    for idx in tqdm(range(len(imgs))):
        # load the image
        img_path = os.path.join(images_path, imgs[idx])
        # read the image, we only read the image to get the size of it
        # so no need to change the format (BGR to RGB) or convert to PIL 
        opencv_img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
        image_height, image_width = opencv_img.shape[:2]
        img_dict = {}
        img_dict["id"] = idx + 1
        img_dict["file_name"] = imgs[idx]
        img_dict["height"] = image_height
        img_dict["width"] = image_width
        json_annotations["images"].append(img_dict)


        # annotations
        boxes: List[List[int]] = []
        labels: List[int] = []
        masks: List[np.ndarray] = []
        # load the annotations (masks)
        annots_path = os.path.join(annotations_path, annotations[idx])
        
        with open(annots_path,'r') as annot_file:
            for line in annot_file:
                fields = line.strip().split(' ')
                if instance_segmentation:
                    # convert the label to an integer from string
                    label = int(fields[0])
                    # convert the polygon points fron strings to floats
                    points = np.array([float(point) for point in fields[1:]])
                    # rearrange them in (x, y)
                    polygon_points = np.reshape(points, (int(len(points) / 2), 2)) * np.array([image_width, image_height], dtype=float)
                    # convert the points to integers and use CV2 contours format
                    polygon_points = np.expand_dims(polygon_points.astype(int), axis=1)
                    # bounding box of the mask contour
                    (xtl, ytl, w, h) = cv2.boundingRect(polygon_points)
                    if w <= 0 or h <= 0:
                        continue
                    xbr = xtl + w 
                    ybr = ytl + h
                    # the mask
                    mask: np.ndarray = np.zeros((image_height, image_width), np.uint8)
                    # create the mask for the object (the mask value is set to 1 for the object)
                    cv2.drawContours(mask, [polygon_points], 0, 1, -1)
                    masks.append(mask)
                else:
                    (label, center_x, center_y, w, h) = fields
                    xtl = int((float(center_x) - float(w) / 2.0) * image_width)
                    ytl = int((float(center_y) - float(h) / 2.0) * image_height)
                    xbr = int((float(center_x) + float(w) / 2.0) * image_width)
                    ybr = int((float(center_y) + float(h) / 2.0) * image_height)
                    if xtl >= xbr or ytl >= ybr:
                        continue
                    # convert the label to an integer from string
                    label = int(label)

                # expand the bounding boxes if necessary
                delta_x = int(percentage_to_expand_bbox_boundaries * (xbr - xtl) / 2)
                delta_y = int(percentage_to_expand_bbox_boundaries * (ybr - ytl) / 2)
                # # expand by one pixel on each side at least to cover boundaries
                delta_x = max(1, delta_x)
                delta_y = max(1, delta_y)
            
                xtl = max(0, xtl - delta_x)
                ytl = max(0, ytl - delta_y)
                xbr = min(image_width, xbr + delta_x)
                ybr = min(image_height, ybr + delta_y)
                boxes.append([xtl, ytl, xbr, ybr])
                labels.append(label)
        
        for i, box in enumerate(boxes):
            record = {}
            record["image_id"] = idx + 1
            record['category_id'] = labels[i] 
            categories.add(record['category_id'])
            if instance_segmentation:
                record["segmentation"] = coco_mask_util.encode(np.asarray(masks[i], order="F"))
                record["segmentation"]['counts'] = record["segmentation"]['counts'].decode('utf8')
            xmin, ymin, xmax, ymax = [int(v) for v in box]
            #convert to xywh
            record['bbox'] = [xmin, ymin, xmax - xmin, ymax - ymin]
            record["area"] = (ymax - ymin) * (xmax - xmin)
            record["iscrowd"] = 0
            record["id"] = annots_id
                
            json_annotations["annotations"].append(record)
            annots_id += 1 
            
    json_annotations["categories"] = [{"id": i} for i in sorted(categories)]

    return True, json_annotations

#### Dataset prepration for option 3
----------------

Define LABEL_MAP, a mapping between class IDs and class names used in the annotations, with class IDs starting from 1  (0 is reseved for background). This is not needed for the dataset, but during the training. 

In [ ]:
# make sure these folders are generated in advance
BASE_PATH = '/home/cellareye/Development/yolov5/data/caging_analysis_cells'
TRAIN_IMAGE_FOLDER = os.path.join(BASE_PATH, 'images', 'train')
TRAIN_MASK_FOLDER = os.path.join(BASE_PATH, 'labels', 'train')
TEST_IMAGE_FOLDER =os.path.join(BASE_PATH, 'images', 'test')
TEST_MASK_FOLDER = os.path.join(BASE_PATH, 'labels', 'test')

# mapping between the class IDs and class names for the annotated data 
# labels start from 0
LABEL_MAP = {0: 'cell', 1: 'bead', 2: 'soma'}
REVERESE_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

train_dataset = CellMaskDataset(images_path=TRAIN_IMAGE_FOLDER, 
                                annotations_path=TRAIN_MASK_FOLDER,
                                instance_segmentation = SEGMENTATION_MODEL,
                                percentage_to_expand_bbox_boundaries = 0.0,
                                transforms = get_transform(train=True))

test_dataset = CellMaskDataset(images_path=TEST_IMAGE_FOLDER, 
                               annotations_path=TEST_MASK_FOLDER,
                               instance_segmentation = SEGMENTATION_MODEL,
                               percentage_to_expand_bbox_boundaries = 0.0,
                               transforms = get_transform(train=False))

In [ ]:
os.listdir(BASE_PATH)

In [ ]:
"""
import json
success, json_annots = convert_to_coco_api(images_path=TEST_IMAGE_FOLDER, 
                                           annotations_path=TEST_MASK_FOLDER, 
                                           instance_segmentation = True)
with open('test_annotations.json', 'w') as file:
    json.dump(json_annots, file)

from references.detection.coco_utils import CocoDetection
# note that this is a modified version of torchvision.datasets.CocoDetection
test_dataset = CocoDetection(img_folder=TEST_IMAGE_FOLDER, ann_file='test_annotations.json', 
                             transforms=get_transform(train=False))
# os.remove('test_annotations.json')
"""

## Model Definition

In [ ]:
from transformers import RTDetrForObjectDetection

def get_rt_detr_model(
    id2label: Dict[int, str], 
):
    pre_trained_model_checkpoint: str = "PekingU/rtdetr_r50vd_coco_o365"
    label2id: Dict[str, int] =  {v: k for k, v in id2label.items()}
    model = RTDetrForObjectDetection.from_pretrained(
        pre_trained_model_checkpoint,
        id2label=id2label,
        label2id=label2id,
        anchor_image_size=None,
        ignore_mismatched_sizes=True,
    )
   
    # this is for freezing the backbone
    # for param in model.model.pixel_level_module.encoder.parameters():
    #     param.requires_grad_(False)

    return model

## Training
### Training parameters

In [ ]:
TRAIN_BATCH_SIZE = 2
OPTIMIZER = 'Adam' # can be set to 'SGD' as well for stochastic Gradient Descent
LEARNING_RATE = 5e-5
NUM_EPOCHS = 4
MODEL_PATH = 'checkpoints'

### Data loaders

It looks like Mask2Former can support overlapping instance masks (the dataset in the tutorial example had non-overlapping instance masks). I need to investigate further to be sure. But it seems, we do not need to define an order of objects when creating the instance masks for overlapping objects. So `collate_fn_2` below should be used. However, if the masks cannot be overlapping, We create the masks of overlapping objects in this order: bg, cage, cell, and then bead as cells can be inside cages (creating holes in cage masks), and beads can potentially be over the cells (creating holes). In this case, `collate_fn_1` should be used.

Note that `test_dataset` below is using a different class (`torchvision.datasets.CocoDetection` instead of `CellMaskDataset` defined above). We do not need any special collate function for the test_data_loader as we only use it for COCO evaluation.

In [ ]:
# we need to resize the input images (not needed as they are already in the correct 640 x 640 input size, and normalize
# them, we use the already implemented Hugging Face preprocessor for this conversion
# we pass all the other flags as False as the image is already augmented 
# index 0 will be used 
from transformers import RTDetrImageProcessor

hg_preprocessor = RTDetrImageProcessor(
    do_convert_annotations=True,
    do_resize=True,
    size={"width": MODEL_INPUT_SIZE, "height": MODEL_INPUT_SIZE},
    reduce_labels=False,
    do_rescale=True, 
    do_normalize=True
)

# it looks like Mas2Former can support overlapping instances, so there is no need to assign each pixel to only one class
# the function collate_fn_1 blow uses the function generate_instance_segmentation that only assigns each pixel to one class, 
# use collate_fn_2 that is more generic
def collate_fn(batch):
    processed_batch = []
    for img, formatted_annotations in batch:
        inputs = hg_preprocessor(images=img, annotations=formatted_annotations, return_tensors="pt")
        inputs = {k: v.squeeze() if isinstance(v, torch.Tensor) else v[0] for k,v in inputs.items()}
        processed_batch.append(inputs)

    data = {}
    data["pixel_values"] = torch.stack([x["pixel_values"] for x in processed_batch])
    data["labels"] = [x["labels"] for x in processed_batch]
    return data

# define training and validation data loaders
train_data_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size = TRAIN_BATCH_SIZE, shuffle = True, num_workers = 2,
    collate_fn = collate_fn)

test_data_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size = 1, shuffle = False, num_workers = 2,
    collate_fn = collate_fn)

# from references.detection.utils import collate_fn as collate_fn_test
# test_data_loader = torch.utils.data.DataLoader(
#     test_dataset, batch_size = 1, shuffle = False, num_workers = 2,
#     collate_fn = collate_fn_test)

print('Training data includes %d annotated images.' %len(train_dataset))
print('Test data includes %d annotated images.' %len(test_dataset))

### Visually checking some data

In [ ]:
import torchvision
COLORS = [(0, 0, 0), (0, 0, 255), (255, 0, 0), (0, 255, 0), (255, 255, 0), (255, 0, 255)]

def show_sample(idx, train = True, instance_segmentation = False):
    # pick the image from the data set
    if train:
        image, annotations = train_dataset[idx]
        boxes = np.array([record['bbox'] for record in annotations['annotations']]).astype(int)
        if len(boxes) > 0:
            boxes[:, 2] += boxes[:, 0]
            boxes[:, 3] += boxes[:, 1]
        labels  = np.array([record['category_id'] for record in annotations['annotations']]).astype(int)
        if instance_segmentation:
            masks = [record['mask'] for record in annotations['annotations']]
    else:
        image, annotations = test_dataset[idx]
        if isinstance(test_dataset, torchvision.datasets.CocoDetection):
            boxes = np.array([t['bbox'] for t in annotations['annotations']])
            if len(boxes) > 0:
                boxes[:, 2] += boxes[:, 0]
                boxes[:, 3] += boxes[:, 1]
            labels = np.array([record['category_id'] for record in annotations['annotations']])
            if instance_segmentation:
                masks = np.array([coco_mask_util.decode(record['segmentation']) for record in annotations['annotations']])
            
        else:
            boxes = np.array([record['bbox'] for record in annotations['annotations']]).astype(int)
            if len(boxes) > 0:
                boxes[:, 2] += boxes[:, 0]
                boxes[:, 3] += boxes[:, 1]
            labels  = np.array([record['category_id'] for record in annotations['annotations']]).astype(int)
            if instance_segmentation:
                masks = [record['mask'] for record in annotations['annotations']]
                
    for i in range(len(labels)):
        # the bounding box
        (xtl, ytl, xbr, ybr) = boxes[i]
        # use green color for masks
        color = COLORS[(labels[i] + 1) % len(COLORS)] # add 1 to be consistent with Mask R-CNN colors/labels
        if instance_segmentation:
            color_mask = color * np.repeat(np.expand_dims(masks[i][ytl:ybr, xtl:xbr], axis=2), 3, axis=2)
            blended = 0.4 * color_mask
            blended[color_mask == 0] = image[ytl:ybr, xtl:xbr][color_mask == 0]
            blended[color_mask > 0] += 0.6 * image[ytl:ybr, xtl:xbr][color_mask > 0]

            # store the blended ROI in the original image
            image[ytl:ybr, xtl:xbr] = blended.astype(np.uint8)
        
        if labels[i] in LABEL_MAP:
            text = LABEL_MAP[labels[i]]
        else:
            print('Incorrect ID was found %s' %labels[i])
            text = 'Unknown'
        
        # add label
        cv2.putText(image, text, (xtl, ytl + 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        # add the bounding box with yellow color
        color = (255, 255, 0)
        cv2.rectangle(image, (xtl, ytl), (xbr, ybr), color, 1)
        
    print(f"Image size (W, H): {image.shape[1]}, {image.shape[0]}")
    # convert to PIL image to display
    return Image.fromarray(image)

In [ ]:
display(show_sample(1, True, SEGMENTATION_MODEL))

### Model selection
### From a pretrained model on COCO (scratch)

In [ ]:
model = get_rt_detr_model(
    id2label=LABEL_MAP
)

# train on the GPU or on the CPU, if a GPU is not available
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

print('Device available:' , device)

# move model to the right device
model.train()
model.to(device)

### Evalutation function
A function to compute COCO mAP and mAR.

In [ ]:
from transformers.image_transforms import center_to_corners_format

def convert_bbox_yolo_to_pascal(boxes, image_size):
    """
    Convert bounding boxes from YOLO format (x_center, y_center, width, height) in range [0, 1]
    to Pascal VOC format (x_min, y_min, x_max, y_max) in absolute coordinates.

    Args:
        boxes (torch.Tensor): Bounding boxes in YOLO format
        image_size (Tuple[int, int]): Image size in format (height, width)

    Returns:
        torch.Tensor: Bounding boxes in Pascal VOC format (x_min, y_min, x_max, y_max)
    """
    # convert center to corners format
    boxes = center_to_corners_format(boxes)

    # convert to absolute coordinates
    height, width = image_size
    boxes = boxes * torch.tensor([[width, height, width, height]])

    return boxes


from dataclasses import dataclass
from torchmetrics.detection.mean_ap import MeanAveragePrecision


@dataclass
class ModelOutput:
    logits: torch.Tensor
    pred_boxes: torch.Tensor


class MAPEvaluator:
    def __init__(self, image_processor, threshold=0.00, id2label=None):
        self.image_processor = image_processor
        self.threshold = threshold
        self.id2label = id2label

    def collect_image_sizes(self, targets):
        """Collect image sizes across the dataset as list of tensors with shape [batch_size, 2]."""
        image_sizes = []
        for batch in targets:
            batch_image_sizes = torch.tensor(np.array([x["size"] for x in batch]))
            image_sizes.append(batch_image_sizes)
        return image_sizes

    def collect_targets(self, targets, image_sizes):
        post_processed_targets = []
        for target_batch, image_size_batch in zip(targets, image_sizes):
            for target, size in zip(target_batch, image_size_batch):
                boxes = torch.tensor(target["boxes"])
                boxes = convert_bbox_yolo_to_pascal(boxes, size)
                labels = torch.tensor(target["class_labels"])
                post_processed_targets.append({"boxes": boxes, "labels": labels})
        return post_processed_targets

    def collect_predictions(self, predictions, image_sizes):
        post_processed_predictions = []
        for batch, target_sizes in zip(predictions, image_sizes):
            batch_logits, batch_boxes = batch[1], batch[2]
            output = ModelOutput(logits=torch.tensor(batch_logits), pred_boxes=torch.tensor(batch_boxes))
            post_processed_output = self.image_processor.post_process_object_detection(
                output, threshold=self.threshold, target_sizes=target_sizes
            )
            post_processed_predictions.extend(post_processed_output)
        return post_processed_predictions

    @torch.no_grad()
    def __call__(self, evaluation_results):

        predictions, targets = evaluation_results.predictions, evaluation_results.label_ids

        image_sizes = self.collect_image_sizes(targets)
        post_processed_targets = self.collect_targets(targets, image_sizes)
        post_processed_predictions = self.collect_predictions(predictions, image_sizes)

        evaluator = MeanAveragePrecision(box_format="xyxy", class_metrics=True)
        evaluator.warn_on_many_detections = False
        evaluator.update(post_processed_predictions, post_processed_targets)

        metrics = evaluator.compute()

        # Replace list of per class metrics with separate metric for each class
        classes = metrics.pop("classes")
        map_per_class = metrics.pop("map_per_class")
        mar_100_per_class = metrics.pop("mar_100_per_class")
        for class_id, class_map, class_mar in zip(classes, map_per_class, mar_100_per_class):
            class_name = id2label[class_id.item()] if id2label is not None else class_id.item()
            metrics[f"map_{class_name}"] = class_map
            metrics[f"mar_100_{class_name}"] = class_mar

        metrics = {k: round(v.item(), 4) for k, v in metrics.items()}

        return metrics

eval_compute_metrics_fn = MAPEvaluator(image_processor=hg_preprocessor, threshold=0.01, id2label=LABEL_MAP)

### Run Training using Hugging face training script

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=MODEL_PATH,
    num_train_epochs=NUM_EPOCHS,
    max_grad_norm=0.1,
    learning_rate=LEARNING_RATE,
    warmup_steps=300,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    dataloader_num_workers=2,
    metric_for_best_model="eval_map",
    greater_is_better=True,
    load_best_model_at_end=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    remove_unused_columns=False,
    eval_do_concat_batches=False,
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=hg_preprocessor,
    data_collator=collate_fn,
    compute_metrics=eval_compute_metrics_fn,
)

trainer.train()

### Run Inference
To run inference on a previously trained model, run the cells above up to "Cell size analysis".

In [ ]:
from torchvision.transforms import functional as F

def to_numpy(tensor):
    """
    A function to convert a torch input to numpy array.
    Args:
        tensor (torch tensor).
    Returns:
        Converted to numpy array.
    """
    return tensor.detach().cpu().numpy() if tensor.requires_grad else tensor.cpu().numpy()

def show_predictions(image_pil, predictions, color_depth=12):
    # convert to a numpy array
    image = np.array(image_pil)
    # scale
    image = (255 * image.astype(float) / (2**color_depth - 1)).astype(np.uint8)
    # convert to 3-channels
    image = np.repeat(np.expand_dims(image, axis=2), 3, axis=2)

    boxes = predictions['boxes']
    labels = predictions['labels']
    masks = predictions['masks']

    for i in range(len(masks)):
        # the bounding box
        (xtl, ytl, xbr, ybr) = boxes[i]
        # use green color for masks
        color = COLORS[labels[i] % len(COLORS)]
        mask = masks[i].copy()
        mask[mask >= 0.3] = 1
        mask[mask < 0.3] = 0
        color_mask = color * np.repeat(np.expand_dims(mask, axis=2), 3, axis=2)
        blended = 0.4 * color_mask
        blended[color_mask == 0] = image[ytl:ybr, xtl:xbr][color_mask == 0]
        blended[color_mask > 0] += 0.6 * image[ytl:ybr, xtl:xbr][color_mask > 0]

        # store the blended ROI in the original image
        image[ytl:ybr, xtl:xbr] = blended.astype(np.uint8)
        
        if labels[i] in LABEL_MAP:
            text = LABEL_MAP[labels[i]]
        else:
            print('Incorrect ID was found %s' %labels[i])
            text = 'Unknown'
        
        # add label
        cv2.putText(image, text, (xtl, ytl + 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        # add the bounding box with yellow color
        color = (255, 255, 0)
        cv2.rectangle(image, (xtl, ytl), (xbr, ybr), color, 1)
        
    # convert to PIL image to display
    return Image.fromarray(image)

# the implementation below is identical to run_model_and_format_results, but we need to call it with a different 
# preprocessor to make sure the image is getting normalized
# we return the results in the same format as Mask R-CNN to be able to reuse the codes written for that model
def predict_batch(model, input_images_list, device):
    
    model.eval()
    model.to(device)

    # convert to 3-channel images if needed, and store the original image dimensions for 
    # post processing
    images_list: List[np.array] = []
    org_img_dims: List[Tuple[int, int]] = []
    
    for img in input_images_list:
        img_shape: tuple = img.shape
        if len(img_shape) < 3:
            images_list.append(cv2.cvtColor(img, cv2.COLOR_GRAY2RGB))
        else:
            images_list.append(img)
        org_img_dims.append(img_shape[:2])
    
    hg_preprocessor = Mask2FormerImageProcessor(ignore_index=0, 
                                                do_resize=True,
                                                size=MODEL_INPUT_SIZE,
                                                size_divisor=14,
                                                reduce_labels=False, 
                                                do_rescale=True,
                                                image_mean=TRANSFORM_MEAN,
                                                image_std=TRANSFORM_STD,
                                                do_normalize=True)
       
    processed_imgs_dict = hg_preprocessor(images_list, return_tensors="pt")
    with torch.no_grad():
        outputs = model(pixel_values=processed_imgs_dict["pixel_values"].to(device))
        processed_outputs = hg_preprocessor.post_process_instance_segmentation(
            outputs, 
            target_sizes=org_img_dims, 
            return_binary_maps=True
        )
   
    if len(processed_outputs) == 0:
        # this should not happen and is not expected, return as if the model has not detected anything (for the whole list of images)
        return [
            {'boxes': [],
             'labels': [],
             'scores': [],
             'masks': []}
        ] * len(images_list)
    
    results = []
    
    for sample_index, processed_output in enumerate(processed_outputs):
        sample_dict = {}
        instance_to_label_map = {segment['id']: segment['label_id'] for segment in processed_output["segments_info"]}
        instance_to_score_map = {segment['id']: segment['score'] for segment in processed_output["segments_info"]}
        sorted_instance_ids = sorted(instance_to_label_map.keys())
        
        if len(sorted_instance_ids) > 0:

            # processed_output['segmentation'] is of dimension num_detections x H x W
            num_instances = processed_output['segmentation'].shape[0]
            
            if num_instances != len(instance_to_label_map):
                print(f"[WARN]: # of instance masks {num_instances} is not equal to the number of labels {len(instance_to_label_map)}!")
                sorted_instance_ids = [i for i in sorted_instance_ids if i < num_instances]

            # masks should be of dimension num_detections x 1 x H x W
            sample_dict['labels'] =  [instance_to_label_map[i] for i in sorted_instance_ids]
            sample_dict['scores'] =  [instance_to_score_map[i] for i in sorted_instance_ids]
            sample_dict['masks'] = to_numpy(processed_output['segmentation'][sorted_instance_ids])
            boxes = []
            masks = [] # masks after restricting them to the size of the bounding box
            for i in range(sample_dict['masks'].shape[0]):
                pos = np.where(sample_dict['masks'][i])
                xtl = pos[1].min()
                xbr = pos[1].max()
                ytl = pos[0].min()
                ybr = pos[0].max()
                boxes.append([xtl, ytl, xbr, ybr])
                masks.append(sample_dict['masks'][i, ytl:ybr, xtl:xbr].astype(float))
            
            sample_dict['boxes'] =  boxes
            sample_dict['masks'] =  masks
            
        else:
            sample_dict = {'boxes': [],
                           'labels': [],
                           'scores': [],
                           'masks': []}
            
        results.append(sample_dict)

    return results

In [ ]:
model = get_mask2former_instance_segmentation_model_with_dinov2_backbone(
    id2label=LABEL_MAP,
    model_type=DINOV2_BACKBONE_TYPE, 
    with_registers=False
)
# load the model, the latest saved checkpoint will be loaded
model.load_state_dict(torch.load(os.path.join(MODEL_PATH, 'checkpoint_' + str(NUM_EPOCHS) + '.pt')))
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
model.eval()

In [ ]:
# idx = 145123
idx = 1412
# img_path = os.path.join(test_dataset.images_path, test_dataset.imgs[idx])
img_id = test_dataset.coco.getImgIds()[idx]
img_path = os.path.join(test_dataset.root, test_dataset.coco.imgs[img_id]['file_name'])
img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)

In [ ]:
out = predict_batch(model, [img], device)[0]

In [ ]:
display(show_predictions(img, out, 8))

In [ ]:
display(show_sample(idx, False))

In [ ]:
bbox_metrics, segm_metrics = evaluate_coco_segm(model, test_data_loader, device=device, max_dets=5000)

In [ ]:
test_dataset_org = CellMaskDataset(images_path=TEST_IMAGE_FOLDER, masks_path=TEST_MASK_FOLDER,
                                   annots_in_coco_rle_format = True,
                                   transforms = get_transform(train=False))
test_data_loader_org = torch.utils.data.DataLoader(
        test_dataset_org, batch_size = 1, shuffle = False, num_workers = 2,
        collate_fn = references.detection.utils.collate_fn)
print('Test data includes %d annotated images.' %len(test_dataset_org))
results = evaluate(model, test_data_loader_org, device=device, max_dets=5000)

### Measuring the run-time

In [ ]:
import time
start = time.time()
for i in range(10):
    out = predict_batch(model, [img], device)[0]
print('Running Mask2Former took {} ms'.format((time.time() - start) * 100))